In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import os

raw_data_path = "/content/drive/MyDrive/ISY 503 Assessment 3"

print(os.listdir(raw_data_path))

['book_negative.review', 'book_positive.review', 'kitch_negative.review', 'kitch_positive.review', 'elec_negative.review', 'elec_positive.review', 'dvd_negative.review', 'dvd_positive.review']


In [7]:
import re
import pandas as pd

raw_data_path = "/content/drive/MyDrive/ISY 503 Assessment 3"

review_files = [
    (f"{raw_data_path}/book_positive.review", "books", 1),
    (f"{raw_data_path}/book_negative.review", "books", 0),

    (f"{raw_data_path}/dvd_positive.review", "dvd", 1),
    (f"{raw_data_path}/dvd_negative.review", "dvd", 0),

    (f"{raw_data_path}/elec_positive.review", "electronics", 1),
    (f"{raw_data_path}/elec_negative.review", "electronics", 0),

    (f"{raw_data_path}/kitch_positive.review", "kitchen_&_housewares", 1),
    (f"{raw_data_path}/kitch_negative.review", "kitchen_&_housewares", 0)
]


def extract_review_text(file_path):
    # Read the complete pseudo-XML file
    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:
        content = file.read()

    # Extract only text contained inside <review_text> tags
    reviews = re.findall(
        r"<review_text>\s*(.*?)\s*</review_text>",
        content,
        flags=re.DOTALL
    )

    return reviews


records = []

for file_path, domain, sentiment in review_files:
    reviews = extract_review_text(file_path)

    for review in reviews:
        records.append({
            "domain": domain,
            "review": review,
            "sentiment": sentiment
        })


raw_df = pd.DataFrame(records)

print("Dataset extracted successfully.")
print(f"Total reviews: {len(raw_df)}")

print("\nReviews by domain and sentiment:")
print(
    raw_df.groupby(
        ["domain", "sentiment"]
    ).size()
)

raw_df.head()

Dataset extracted successfully.
Total reviews: 8000

Reviews by domain and sentiment:
domain                sentiment
books                 0            1000
                      1            1000
dvd                   0            1000
                      1            1000
electronics           0            1000
                      1            1000
kitchen_&_housewares  0            1000
                      1            1000
dtype: int64


,domain,review,sentiment
0,books,Sphere by Michael Crichton is an excellant nov...,1
1,books,Dr. Oz is an accomplished heart surgeon in the...,1
2,books,The most gorgeous artwork in comic books. Cont...,1
3,books,This book is for lovers of Robicheaux. His de...,1
4,books,This is going to be a short and sweet review b...,1


In [8]:
# CLEAN REVIEW TEXT

import re

# Create a working copy
df = raw_df.copy()


def clean_review(text):
    """
    Clean review text while preserving the words
    that may carry sentiment information.
    """

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Keep alphabetic characters and apostrophes
    # Apostrophes preserve expressions such as don't and wasn't
    text = re.sub(r"[^a-zA-Z']", " ", text)

    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning
df["review"] = df["review"].apply(clean_review)

print("Text cleaning completed successfully.")

df[["domain", "review", "sentiment"]].head()

Text cleaning completed successfully.


,domain,review,sentiment
0,books,sphere by michael crichton is an excellant nov...,1
1,books,dr oz is an accomplished heart surgeon in the ...,1
2,books,the most gorgeous artwork in comic books conta...,1
3,books,this book is for lovers of robicheaux his demo...,1
4,books,this is going to be a short and sweet review b...,1


In [9]:
# CHECK CLEANED REVIEW QUALITY

# Calculate review length after cleaning
df["review_length"] = df["review"].apply(
    lambda x: len(x.split())
)

print("Cleaned Review Statistics")
print("-" * 35)

print(f"Total reviews       : {len(df)}")
print(f"Empty reviews       : {(df['review_length'] == 0).sum()}")
print(f"1-word reviews      : {(df['review_length'] == 1).sum()}")
print(f"2-word reviews      : {(df['review_length'] == 2).sum()}")
print(f"Minimum length      : {df['review_length'].min()} words")
print(f"Maximum length      : {df['review_length'].max()} words")
print(f"Average length      : {df['review_length'].mean():.2f} words")
print(f"Median length       : {df['review_length'].median():.0f} words")

Cleaned Review Statistics
-----------------------------------
Total reviews       : 8000
Empty reviews       : 0
1-word reviews      : 1
2-word reviews      : 0
Minimum length      : 1 words
Maximum length      : 3450 words
Average length      : 135.21 words
Median length       : 90 words


In [10]:
rows_before = len(df)

# Keep reviews containing at least 3 words
df = df[df["review_length"] >= 3].copy()

# Reset dataframe index
df.reset_index(drop=True, inplace=True)

rows_after = len(df)

print("Short Review Removal")
print("-" * 35)

print(f"Rows before removal : {rows_before}")
print(f"Reviews removed     : {rows_before - rows_after}")
print(f"Rows remaining      : {rows_after}")
print(f"Minimum length      : {df['review_length'].min()} words")

Short Review Removal
-----------------------------------
Rows before removal : 8000
Reviews removed     : 1
Rows remaining      : 7999
Minimum length      : 4 words


In [11]:
duplicate_count = df["review"].duplicated().sum()

print("Duplicate Review Check")
print("-" * 35)

print(f"Total reviews     : {len(df)}")
print(f"Duplicate reviews : {duplicate_count}")
print(f"Unique reviews    : {df['review'].nunique()}")

Duplicate Review Check
-----------------------------------
Total reviews     : 7999
Duplicate reviews : 149
Unique reviews    : 7850


In [12]:
rows_before = len(df)

# Remove duplicate review text
df = df.drop_duplicates(
    subset="review",
    keep="first"
).reset_index(drop=True)

rows_after = len(df)

print("Duplicate Removal")
print("-" * 35)

print(f"Rows before removal     : {rows_before}")
print(f"Duplicates removed      : {rows_before - rows_after}")
print(f"Rows remaining          : {rows_after}")
print(f"Duplicates remaining    : {df['review'].duplicated().sum()}")

Duplicate Removal
-----------------------------------
Rows before removal     : 7999
Duplicates removed      : 149
Rows remaining          : 7850
Duplicates remaining    : 0


In [13]:
sentiment_counts = df["sentiment"].value_counts().sort_index()
sentiment_percentages = (
    df["sentiment"].value_counts(normalize=True)
    .sort_index() * 100
)

print("Final Dataset")
print("-" * 35)
print(f"Total reviews: {len(df)}")

print("\nSentiment Distribution")
print("-" * 35)

print(f"Negative (0): {sentiment_counts[0]} "
      f"({sentiment_percentages[0]:.2f}%)")

print(f"Positive (1): {sentiment_counts[1]} "
      f"({sentiment_percentages[1]:.2f}%)")

Final Dataset
-----------------------------------
Total reviews: 7850

Sentiment Distribution
-----------------------------------
Negative (0): 3880 (49.43%)
Positive (1): 3970 (50.57%)


In [14]:
# SPLIT DATA INTO TRAINING, VALIDATION AND TEST SETS

from sklearn.model_selection import train_test_split

# First split:
# 70% training and 30% temporary data
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["sentiment"]
)

# Second split:
# Divide temporary data equally into validation and test sets
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["sentiment"]
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Dataset Split")
print("-" * 35)

print(f"Training samples   : {len(train_df)}")
print(f"Validation samples : {len(val_df)}")
print(f"Test samples       : {len(test_df)}")
print(f"Total samples      : {len(train_df) + len(val_df) + len(test_df)}")

Dataset Split
-----------------------------------
Training samples   : 5495
Validation samples : 1177
Test samples       : 1178
Total samples      : 7850


In [15]:

# VERIFY CLASS DISTRIBUTION AFTER SPLITTING


def show_distribution(name, data):

    percentages = (
        data["sentiment"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )

    print(name)
    print(f"Negative (0): {percentages[0]:.2f}%")
    print(f"Positive (1): {percentages[1]:.2f}%")
    print("-" * 30)


show_distribution("Training Set", train_df)
show_distribution("Validation Set", val_df)
show_distribution("Test Set", test_df)

Training Set
Negative (0): 49.43%
Positive (1): 50.57%
------------------------------
Validation Set
Negative (0): 49.45%
Positive (1): 50.55%
------------------------------
Test Set
Negative (0): 49.41%
Positive (1): 50.59%
------------------------------


In [17]:
# CREATE AND FIT TOKENIZER

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenisation parameters
VOCAB_SIZE = 20000
MAX_LENGTH = 100
OOV_TOKEN = "<OOV>"

# Create tokenizer
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN
)

# IMPORTANT:
# Fit vocabulary using training reviews only
tokenizer.fit_on_texts(
    train_df["review"]
)

print("Tokenizer fitted successfully.")
print(f"Full vocabulary size : {len(tokenizer.word_index) + 1}")
print(f"Model vocabulary size: {VOCAB_SIZE}")

Tokenizer fitted successfully.
Full vocabulary size : 33790
Model vocabulary size: 20000


In [18]:
# Convert review text into integer sequences
train_sequences = tokenizer.texts_to_sequences(train_df["review"])
val_sequences = tokenizer.texts_to_sequences(val_df["review"])
test_sequences = tokenizer.texts_to_sequences(test_df["review"])

# Pad or truncate every review to 100 tokens
X_train = pad_sequences(
    train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_val = pad_sequences(
    val_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

# Convert sentiment labels to numerical arrays
y_train = train_df["sentiment"].to_numpy()
y_val = val_df["sentiment"].to_numpy()
y_test = test_df["sentiment"].to_numpy()

print("Tokenisation and padding completed successfully.")

print("\nInput Shapes")
print("-" * 35)
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\nLabel Shapes")
print("-" * 35)
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

print("\nSequence length:", MAX_LENGTH)

Tokenisation and padding completed successfully.

Input Shapes
-----------------------------------
X_train: (5495, 100)
X_val  : (1177, 100)
X_test : (1178, 100)

Label Shapes
-----------------------------------
y_train: (5495,)
y_val  : (1177,)
y_test : (1178,)

Sequence length: 100


In [19]:
# SAVE TOKENIZER FOR FUTURE INFERENCE

import pickle
import os

save_path = "/content/drive/MyDrive/ISY 503 Assessment 3"

tokenizer_path = os.path.join(
    save_path,
    "sentiment_tokenizer.pkl"
)

# Save tokenizer
with open(tokenizer_path, "wb") as file:
    pickle.dump(tokenizer, file)

print("Tokenizer saved successfully.")
print("Location:", tokenizer_path)

# Confirm file exists
print("File exists:", os.path.exists(tokenizer_path))

Tokenizer saved successfully.
Location: /content/drive/MyDrive/ISY 503 Assessment 3/sentiment_tokenizer.pkl
File exists: True


In [20]:
# CREATE TENSORFLOW DATASETS

import tensorflow as tf

BATCH_SIZE = 32

# Create datasets
train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test)
)

# Shuffle training data only, then create batches
train_dataset = (
    train_dataset
    .shuffle(
        buffer_size=len(X_train),
        seed=42
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Validation and test sets should NOT be shuffled
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("TensorFlow datasets created successfully.")

print("\nNumber of Batches")
print("-" * 35)

print("Training   :", tf.data.experimental.cardinality(train_dataset).numpy())
print("Validation :", tf.data.experimental.cardinality(val_dataset).numpy())
print("Test       :", tf.data.experimental.cardinality(test_dataset).numpy())

print("\nBatch size :", BATCH_SIZE)

TensorFlow datasets created successfully.

Number of Batches
-----------------------------------
Training   : 172
Validation : 37
Test       : 37

Batch size : 32


In [21]:
# MODEL 1 - GLOBAL AVERAGE POOLING


from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

# Reproducibility
tf.keras.utils.set_random_seed(42)

EMBEDDING_DIM = 64

model_1 = Sequential([

    # Explicit input shape
    tf.keras.Input(shape=(MAX_LENGTH,)),

    # Convert token IDs into learned word vectors
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True
    ),

    # Produce one representation for each review
    GlobalAveragePooling1D(),

    # Hidden layer
    Dense(64, activation="relu"),

    # Reduce overfitting
    Dropout(0.5),

    # Binary sentiment prediction
    Dense(1, activation="sigmoid")
])

print("Model 1 created successfully.")

model_1.summary()

Model 1 created successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,284,225 (4.90 MB)

 Trainable params: 1,284,225 (4.90 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# COMPILE MODEL 1

model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

print("Model 1 compiled successfully.")

Model 1 compiled successfully.


In [23]:
# CONFIGURE EARLY STOPPING

early_stopping_1 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("Early stopping configured successfully.")

Early stopping configured successfully.


In [24]:
# TRAIN MODEL 1

history_model_1 = model_1.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[early_stopping_1],
    verbose=1
)

Epoch 1/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.6684 - loss: 0.6471 - precision: 0.6561 - recall: 0.7236 - val_accuracy: 0.7596 - val_loss: 0.5379 - val_precision: 0.7776 - val_recall: 0.7345
Epoch 2/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.8444 - loss: 0.3921 - precision: 0.8532 - recall: 0.8363 - val_accuracy: 0.7893 - val_loss: 0.4511 - val_precision: 0.7794 - val_recall: 0.8134
Epoch 3/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9252 - loss: 0.2165 - precision: 0.9318 - recall: 0.9194 - val_accuracy: 0.7825 - val_loss: 0.4890 - val_precision: 0.7712 - val_recall: 0.8101
Epoch 4/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9672 - loss: 0.1213 - precision: 0.9700 - recall: 0.9651 - val_accuracy: 0.7799 - val_loss: 0.5483 - val_precision: 0.7791 - val_recall: 0.7882
Epoch 5/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.9854 - loss: 0.0688 - precision: 0.9856 - recall: 0.9856 - val_accuracy: 0.7893 - val_los

In [25]:
# EVALUATE MODEL 1 ON TEST DATA
model_1_results = model_1.evaluate(
    test_dataset,
    verbose=1
)

test_loss_1 = model_1_results[0]
test_accuracy_1 = model_1_results[1]
test_precision_1 = model_1_results[2]
test_recall_1 = model_1_results[3]

print("\nModel 1 Test Performance")
print("-" * 35)

print(f"Test Loss      : {test_loss_1:.4f}")
print(f"Test Accuracy  : {test_accuracy_1:.4f}")
print(f"Test Precision : {test_precision_1:.4f}")
print(f"Test Recall    : {test_recall_1:.4f}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7767 - loss: 0.4470 - precision: 0.7770 - recall: 0.7836

Model 1 Test Performance
-----------------------------------
Test Loss      : 0.4470
Test Accuracy  : 0.7767
Test Precision : 0.7770
Test Recall    : 0.7836


In [26]:
# MODEL 2 - BIDIRECTIONAL LSTM

from tensorflow.keras.layers import LSTM, Bidirectional

# Reproducibility
tf.keras.utils.set_random_seed(42)

model_2 = Sequential([

    tf.keras.Input(shape=(MAX_LENGTH,)),

    # Word embeddings
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=64,
        mask_zero=True
    ),

    # Learn sequential information in both directions
    Bidirectional(
        LSTM(64)
    ),

    # Regularisation
    Dropout(0.5),

    # Hidden layer
    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.3),

    # Binary classification
    Dense(
        1,
        activation="sigmoid"
    )
])

print("Model 2 created successfully.")

model_2.summary()

Model 2 created successfully.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,350,209 (5.15 MB)

 Trainable params: 1,350,209 (5.15 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:

model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

print("Model 2 compiled successfully.")

Model 2 compiled successfully.


In [28]:
early_stopping_2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("Early stopping configured successfully.")

Early stopping configured successfully.


In [29]:
history_model_2 = model_2.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[early_stopping_2],
    verbose=1
)

Epoch 1/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 39s 180ms/step - accuracy: 0.6195 - loss: 0.6403 - precision: 0.6055 - recall: 0.7103 - val_accuracy: 0.7587 - val_loss: 0.5019 - val_precision: 0.7832 - val_recall: 0.7227
Epoch 2/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 30s 176ms/step - accuracy: 0.8590 - loss: 0.3471 - precision: 0.8657 - recall: 0.8535 - val_accuracy: 0.7749 - val_loss: 0.4773 - val_precision: 0.7705 - val_recall: 0.7899
Epoch 3/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 32s 184ms/step - accuracy: 0.9500 - loss: 0.1427 - precision: 0.9504 - recall: 0.9507 - val_accuracy: 0.7791 - val_loss: 0.5797 - val_precision: 0.8007 - val_recall: 0.7496
Epoch 4/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 32s 188ms/step - accuracy: 0.9807 - loss: 0.0646 - precision: 0.9816 - recall: 0.9802 - val_accuracy: 0.7672 - val_loss: 0.8890 - val_precision: 0.7520 - val_recall: 0.8050
Epoch 5/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 39s 178ms/step - accuracy: 0.9869 - loss: 0.0417 - precision: 0.9881 - recall: 0.9860 - val_accuracy: 0.7502

In [30]:
# EVALUATE MODEL 2 ON TEST DATA
model_2_results = model_2.evaluate(
    test_dataset,
    verbose=1
)

test_loss_2 = model_2_results[0]
test_accuracy_2 = model_2_results[1]
test_precision_2 = model_2_results[2]
test_recall_2 = model_2_results[3]

print("\nModel 2 Test Performance")
print("-" * 35)

print(f"Test Loss      : {test_loss_2:.4f}")
print(f"Test Accuracy  : {test_accuracy_2:.4f}")
print(f"Test Precision : {test_precision_2:.4f}")
print(f"Test Recall    : {test_recall_2:.4f}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy: 0.7623 - loss: 0.4770 - precision: 0.7642 - recall: 0.7668

Model 2 Test Performance
-----------------------------------
Test Loss      : 0.4770
Test Accuracy  : 0.7623
Test Precision : 0.7642
Test Recall    : 0.7668


In [31]:
# SAVE FINAL MODEL


import os

model_path = (
    "/content/drive/MyDrive/ISY 503 Assessment 3/"
    "sentiment_model.keras"
)

model_1.save(model_path)

print("Final model saved successfully.")
print("Location:", model_path)
print("File exists:", os.path.exists(model_path))

Final model saved successfully.
Location: /content/drive/MyDrive/ISY 503 Assessment 3/sentiment_model.keras
File exists: True


In [33]:
# SENTIMENT PREDICTION FUNCTION


def predict_sentiment(text):

    # Clean the new review using the same preprocessing
    cleaned_text = clean_review(text)

    # Convert text to token sequence
    sequence = tokenizer.texts_to_sequences([cleaned_text])

    # Pad/truncate to the same length used during training
    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    # Predict probability of positive sentiment
    probability = model_1.predict(
        padded_sequence,
        verbose=0
    )[0][0]

    # Convert probability into a sentiment label
    if probability >= 0.5:
        sentiment = "Positive review"
    else:
        sentiment = "Negative review"

    return sentiment, float(probability)

In [34]:
test_reviews = [
    "This product is excellent and works perfectly. I highly recommend it.",
    "This was a terrible purchase. It stopped working after two days.",
    "The quality is very good and I am extremely happy with it.",
    "I regret buying this product. It is poorly made and useless."
]

for review in test_reviews:

    sentiment, probability = predict_sentiment(review)

    print("Review:", review)
    print("Prediction:", sentiment)
    print(f"Positive probability: {probability:.4f}")
    print("-" * 80)

Review: This product is excellent and works perfectly. I highly recommend it.
Prediction: Positive review
Positive probability: 1.0000
--------------------------------------------------------------------------------
Review: This was a terrible purchase. It stopped working after two days.
Prediction: Negative review
Positive probability: 0.0013
--------------------------------------------------------------------------------
Review: The quality is very good and I am extremely happy with it.
Prediction: Positive review
Positive probability: 0.9940
--------------------------------------------------------------------------------
Review: I regret buying this product. It is poorly made and useless.
Prediction: Negative review
Positive probability: 0.0132
--------------------------------------------------------------------------------
